# 🧪 W2-D6 概念实验：QLoRA 微调

> 配套阅读：`第2周-Day6-QLoRA微调实战.md`（完整流程、NF4 原理、业务关联在那边）
> 这个 notebook 用可执行实验回答三个问题：
> 1. **NF4 量化比均匀 int4 好在哪？** 为什么正态分布权重用 NF4 误差更小？
> 2. **LoRA 的低秩近似真的够用吗？** SVD 验证权重变化的低秩性
> 3. **QLoRA 显存到底省了多少？** 全量微调 vs LoRA vs QLoRA 的内存账

## 实验 1：NF4 vs 均匀 int4 — 为什么正态分布权重要用 NF4？

模型权重近似正态分布。NF4 的量化级别按正态分布的分位数排列，在概率密度高的区域（接近 0）分配更精细的级别。均匀量化则在所有区间等距分配。

In [ ]:
import numpy as np
from math import erf, sqrt, log

np.random.seed(42)
n = 100000
weights = np.random.randn(n) * 0.4

# 手动实现 erfinv（仅用于 [0,1] 范围）
def erfinv(x):
    """Inverse error function (approximation, sufficient for demo)"""
    a = [0.8862269255, -1.6453436556, 0.9803505318, -1.1975498636]
    b = [-2.0689166073, -3.5457443498, 1.2986744981]
    p = 0.0705280784
    sign = 1 if x >= 0 else -1
    x = abs(x)
    if x <= p:
        r = ((((a[3]*x + a[2])*x + a[1])*x + a[0])*x)
    elif x < 1 - p:
        r = sqrt(-2*log(1-x))
        r = ((((b[2]*r + b[1])*r + b[0])*r + a[2])*r + a[1])*r + a[0]
    else:
        r = float('inf')
    return sign * r

# 均匀 int4 对称量化
def uniform_quantize(x, n_bits=4, vmax=1.0):
    levels = 2 ** n_bits
    scale = 2 * vmax / (levels - 1)
    x_clip = np.clip(x, -vmax, vmax)
    x_q = np.round(x_clip / scale)
    return x_q * scale

# NF4 量化（级别 = 正态分位数中点）
def nf4_quantize(x, n_bits=4):
    levels = 2 ** n_bits
    # 量化级别：正态 CDF 等概率分位数的中点
    edges = []
    for i in range(levels + 1):
        q = i / levels
        if q == 0:
            edges.append(-2.5)
        elif q == 1:
            edges.append(2.5)
        else:
            edges.append(sqrt(2) * erfinv(2 * q - 1))
    edges = np.array(edges)
    midpoints = (edges[:-1] + edges[1:]) / 2

    x_clip = np.clip(x, edges[0], edges[-1])
    indices = np.searchsorted(midpoints, x_clip)
    indices = np.clip(indices, 0, levels - 1)
    return midpoints[indices]

w_uniform = uniform_quantize(weights)
w_nf4 = nf4_quantize(weights)

mse_uniform = np.mean((weights - w_uniform) ** 2)
mse_nf4 = np.mean((weights - w_nf4) ** 2)

print(f"量化误差对比（{n:,} 个正态分布权重，4-bit）")
print(f"{'方法':>15} {'MSE':>12} {'Max Error':>12}")
print(f"{'均匀 INT4':>15} {mse_uniform:>12.6f} {np.max(np.abs(weights - w_uniform)):>12.6f}")
print(f"{'NF4':>15} {mse_nf4:>12.6f} {np.max(np.abs(weights - w_nf4)):>12.6f}")
print(f"\nNF4 比 INT4 均匀量化 MSE 降低了 {(1-mse_nf4/mse_uniform)*100:.1f}%")
print("原因：NF4 在概率密度高的区域（接近 0）分配了更精细的量化级别。")

## 实验 2：LoRA 的低秩假设 — 微调权重的 SVD 分析

LoRA 假设微调引起的变化量 ΔW 是低秩的。用 SVD 验证。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(0)
d = 256
rank_true = 4

# 模拟低秩微调变化 ΔW = A @ B (rank=4)
A = np.random.randn(d, rank_true) * 0.01
B = np.random.randn(rank_true, d) * 0.01
delta_W = A @ B

U, s, Vt = np.linalg.svd(delta_W, full_matrices=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 左图：奇异值谱
axes[0].bar(range(min(20, len(s))), s[:20], color='steelblue')
axes[0].axvline(rank_true - 0.5, color='red', ls='--', label=f'真实 rank={rank_true}')
axes[0].set_xlabel('奇异值索引')
axes[0].set_ylabel('奇异值')
axes[0].set_title('ΔW 的奇异值谱（只有前 4 个非零）')
axes[0].legend()

# 右图：不同 rank 重建的误差
ranks = range(1, 33)
errors = []
for r in ranks:
    W_approx = U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]
    errors.append(np.linalg.norm(delta_W - W_approx) / np.linalg.norm(delta_W))

axes[1].plot(list(ranks), errors, 'o-', markersize=4)
axes[1].axvline(rank_true, color='red', ls='--', alpha=0.7, label=f'真实 rank={rank_true}')
axes[1].set_xlabel('LoRA rank (r)')
axes[1].set_ylabel('相对误差')
axes[1].set_title('LoRA rank 越大，近似越精确')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# LoRA 参数量
d_model = 4096
print("LoRA 参数量 vs 原始权重（d_model=4096）:")
for r in [4, 8, 16, 32, 64]:
    lora_p = 2 * d_model * r
    orig_p = d_model * d_model
    print(f"  rank={r:>3}: LoRA 参数 {lora_p/1e6:.2f}M ({lora_p/orig_p*100:.2f}% of W)")

## 实验 3：QLoRA 显存账 — 全量微调 vs LoRA vs QLoRA

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

param_count = 7e9

# 全量微调 (fp16 + Adam fp32 states)
full_finetune_gb = param_count * 2 / 1e9 + param_count * 8 / 1e9 + param_count * 2 / 1e9

# LoRA (fp16 冻结权重 + fp16 LoRA + Adam on LoRA only)
lora_ratio = 0.008
lora_gb = param_count * 2 / 1e9 + (param_count * lora_ratio * 2 / 1e9 + param_count * lora_ratio * 8 / 1e9 + param_count * lora_ratio * 2 / 1e9)

# QLoRA (NF4 冻结权重 + fp16 LoRA + Adam on LoRA)
qlora_gb = param_count * 0.5 / 1e9 + (param_count * lora_ratio * 2 / 1e9 + param_count * lora_ratio * 8 / 1e9 + param_count * lora_ratio * 2 / 1e9)

# 纯推理 NF4
infer_gb = param_count * 0.5 / 1e9

schemes = [full_finetune_gb, lora_gb, qlora_gb, infer_gb]
labels = ['全量微调\nfp16+Adam', 'LoRA\nfp16+LoRA', 'QLoRA\nNF4+LoRA', '纯推理\nNF4']
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, schemes, color=colors, width=0.6, edgecolor='white')
for bar, val in zip(bars, schemes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f} GB', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(80, color='gray', ls='--', alpha=0.4, label='A100 80GB')
ax.axhline(24, color='gray', ls=':', alpha=0.4, label='RTX 4090 24GB')
ax.axhline(16, color='gray', ls='-.', alpha=0.4, label='RTX 4060Ti 16GB')
ax.set_ylabel('显存占用 (GB)')
ax.set_title('7B 模型微调：不同方案的显存需求')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f"全量微调: {full_finetune_gb:.1f} GB")
print(f"LoRA:      {lora_gb:.1f} GB")
print(f"QLoRA:     {qlora_gb:.1f} GB")
print(f"纯推理:    {infer_gb:.1f} GB")
print(f"\nQLoRA 节省 {(1 - qlora_gb/full_finetune_gb)*100:.0f}% 显存")
print(f"QLoRA 只需 ~{qlora_gb:.0f} GB → 消费级 GPU 即可微调 7B 模型！")